# Curanex: survey screeners round 3 - threshold set on one NHANES cycle, tested on another (CPU)

Round 2 models (BRFSS 2022-2025 + education; skin cancer also white vs not) rank well on NHANES 2015-16 (AUC 0.913 skin cancer,
0.817 cancer, 0.798 vision loss), but the BRFSS threshold does not carry over. **Rules fixed before running (2026-11-11):**
models unchanged; the flagging threshold is set once on NHANES 2017-18 adults (highest with sensitivity >= 0.85 there);
**outside test, scored once:** NHANES 2013-14 adults, never used for these models. Verified if sensitivity and AUC are both
>= 0.80 on 2013-14; live at >= 0.75; otherwise off.

In [ ]:
import io, json, glob, urllib.request
import numpy as np, pandas as pd
from sklearn.metrics import roc_auc_score
AGE_MID = {1: 21, 2: 27, 3: 32, 4: 37, 5: 42, 6: 47, 7: 52, 8: 57, 9: 62, 10: 67, 11: 72, 12: 77, 13: 82}
yes = lambda s: s.map({1.0: 1.0, 2.0: 0.0})


def load(YEAR, L):
    def nh(name):
        url = f"https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/{YEAR}/DataFiles/{name}_{L}.xpt"
        try:
            return pd.read_sas(io.BytesIO(urllib.request.urlopen(url, timeout=120).read()), format="xport")
        except Exception as e:
            print("missing", name, L, e); return pd.DataFrame({"SEQN": []})


    d = nh("DEMO")[["SEQN", "RIAGENDR", "RIDAGEYR", "DMDEDUC2", "RIDRETH3"]]
    for name in ("BMX", "SMQ", "PAQ", "HSQ", "DLQ", "DIQ", "MCQ", "BPQ", "KIQ_U"):
        d = d.merge(nh(name), on="SEQN", how="left")
    col = lambda c: d[c] if c in d else pd.Series(np.nan, index=d.index)
    d = d[d.RIDAGEYR >= 18]
    age_bin = np.clip(((d.RIDAGEYR - 25) // 5 + 2).where(d.RIDAGEYR >= 25, 1), 1, 13)
    smk_now = np.where(col("SMQ020") == 2, 0.0, col("SMQ040").map({1.0: 1.0, 2.0: 1.0, 3.0: 0.0}))
    cancer_types = [c for c in ("MCQ230A", "MCQ230B", "MCQ230C", "MCQ230D") if c in d]
    types = d[cancer_types]
    skin = types.isin([31, 32]).any(axis=1)
    other = (types.notna() & ~types.isin([31, 32, 77, 99])).any(axis=1)
    had = col("MCQ220")
    nhanes = pd.DataFrame({
        "age": age_bin.map(AGE_MID), "male": col("RIAGENDR").map({1.0: 1.0, 2.0: 0.0}), "bmi": col("BMXBMI"),
        "smoker_current": smk_now, "smoker_former": np.where((col("SMQ020") == 1) & (col("SMQ040") == 3), 1.0, np.where(col("SMQ020").isin([1, 2]), 0.0, np.nan)),
        "exercise": np.where((col("PAQ650") == 1) | (col("PAQ665") == 1), 1.0, np.where((col("PAQ650") == 2) & (col("PAQ665") == 2), 0.0, np.nan)),
        "general_health": col("HSD010").where(col("HSD010") <= 5), "poor_physical_days": col("HSQ470").where(col("HSQ470") <= 30),
        "poor_mental_days": col("HSQ480").where(col("HSQ480") <= 30), "difficulty_walking": yes(col("DLQ050")), "difficulty_concentrating": yes(col("DLQ040")),
        "diabetes": col("DIQ010").map({1.0: 1.0, 2.0: 0.0, 3.0: 0.0}), "heart_disease": np.fmax(yes(col("MCQ160C")), yes(col("MCQ160E"))),
        "stroke": yes(col("MCQ160F")), "asthma": yes(col("MCQ010")), "copd": yes(col("MCQ160O")), "kidney_disease": yes(col("KIQ022")),
        "arthritis": yes(col("MCQ160A")),
        "skin_cancer": np.where(had == 2, 0.0, np.where((had == 1) & skin, 1.0, np.nan)),
        "cancer": np.where(had == 2, 0.0, np.where((had == 1) & other, 1.0, np.nan)),
        "education": col("DMDEDUC2").map({1.0: 1.0, 2.0: 1.0, 3.0: 2.0, 4.0: 3.0, 5.0: 4.0}),
        "white": col("RIDRETH3").map(lambda v: float(v == 3) if v == v else np.nan),
        "vision_loss": yes(col("DLQ020")), "hearing_difficulty": yes(col("DLQ010")), "high_blood_pressure": yes(col("BPQ020")),
    })
    return nhanes


cal, test = load(2017, "J"), load(2013, "H")
print("NHANES 2017-18:", len(cal), "| 2013-14:", len(test), {t: int(test[t].sum()) for t in ("skin_cancer", "cancer", "vision_loss")})

In [ ]:
def proba(meta, X):
    raw = np.full(len(X), meta["baseline"])
    for t in meta["trees"]:
        fi, th, le, ri, leaf, val, ml = (np.array(t[k]) for k in ("feature_idx", "num_threshold", "left", "right", "is_leaf", "value", "missing_go_to_left"))
        n = np.zeros(len(X), int)
        for _ in range(64):
            act = ~leaf[n].astype(bool)
            if not act.any():
                break
            v = X[np.arange(len(X)), fi[n]]
            go = np.where(np.isnan(v), ml[n].astype(bool), v <= th[n])
            n = np.where(act, np.where(go, le[n], ri[n]), n)
        raw += val[n]
    return 1 / (1 + np.exp(-raw))


def report(y, p, t):
    f = p >= t
    r = {"n": int(len(y)), "positives": int(y.sum()), "sensitivity": round(float(f[y == 1].mean()), 3),
         "specificity": round(float((~f[y == 0]).mean()), 3), "auc": round(float(roc_auc_score(y, p)), 3)}
    low = min(r["sensitivity"], r["auc"]); r["status"] = "verified" if low >= 0.80 else "live" if low >= 0.75 else "off"
    return r


out = {}
for t in ("skin_cancer", "cancer", "vision_loss"):
    meta = json.load(open(glob.glob(f"/kaggle/input/**/{t}.json", recursive=True)[0]))
    assert "trees" in meta, t
    c, b = cal[cal[t].notna()], test[test[t].notna()]
    pc = proba(meta, c[meta["features"]].to_numpy(np.float64)); yc = c[t].to_numpy(int)
    thr = float(max(v for v in np.unique(np.round(pc, 6)) if (pc[yc == 1] >= v).mean() >= 0.85))
    res = report(b[t].to_numpy(int), proba(meta, b[meta["features"]].to_numpy(np.float64)), thr)
    print(f"OUTSIDE TEST on NHANES 2013-14 (once): {t} threshold {thr:.4f} (set on 2017-18: {report(yc, pc, thr)}) -> {res}", flush=True)
    meta["threshold"] = thr
    meta["metrics"]["calibration_2017"] = report(yc, pc, thr)
    meta["metrics"]["test_nhanes_2013"] = res
    json.dump(meta, open(f"/kaggle/working/{t}.json", "w"))
    out[t] = res
json.dump(out, open("/kaggle/working/summary.json", "w"), indent=1)